In [ ]:
from fuzzywuzzy import fuzz
import re

CUTOFF_THRES = 70
POS_THRES = 5

def get_closest(key, words, cutoff):
    sim = []
    for word in words:
        if fuzz.token_sort_ratio(key, word) > cutoff:
            sim.append(word)
    return sim


def get_index(s, word):
    '''get the possible match positions of word in sentence s'''
    cutoff_t = CUTOFF_THRES
    match_index = []
    words = s.split(' ')
    
    if len(word) < 5:
        cutoff_t = 90
        
    matches = get_closest(word, words, cutoff=cutoff_t)
    
    if len(matches) > 0:
        for match in matches:
            match_index.append((match, words.index(match)))
    return list(set(match_index))


def judge_distance(pos1, pos2, l):
    '''return whether pos1 is in front of pos2 and the distance is less than l'''
    return pos1 - pos2 > 0 and pos1 - pos2 <= l

keys = {
    'order_key': ['switch', 'replace', 'migrate', 'swap', 'upgrade']
}

def is_migration(p, old_lib, new_lib):
    '''
    judge whether p document a migration from old_lib to new_lib
    
    return True or False, key sentence, rule
    '''
    p = p.lower()
    # print(p)
    for s in p.split('.'): # can be replaced with sentence segmentation, word segmentation, part-of-speech tagging, lemmatization, here just split by '.'
        new_lib_poss = get_index(s, new_lib)
        old_lib_poss = get_index(s, old_lib)
        # key old_lib  new_lib  # replace old_lib with new_lib, switch old_lib to new_lib
        order_key = ['switch', 'replace', 'migrate', 'swap', 'upgrade']
        key_pos = [
            item
            for sublist in [get_index(s, key) for key in order_key]
            for item in sublist
        ]
        for p in key_pos:
            for new_lib_pos in new_lib_poss:
                for old_lib_pos in old_lib_poss:
                    if p[1] < old_lib_pos[1] and old_lib_pos[
                            1] < new_lib_pos[1] and judge_distance(
                                old_lib_pos[1], p[1], POS_THRES) and judge_distance(
                                    new_lib_pos[1], old_lib_pos[1], POS_THRES):
                        return True, s, 'key B A'
                    
        # other patterns
        # ...           
                    
                    
    return False, '', ''

def split_lib_name_c(lib: str):
    '''split library name into parts and delete useless of them'''
    delete_parts = ['devel', 'core', 'modules', 'dev', 'lib']
    lib = ' '.join([re.search('(.*)(?:\[.*\])*(?:\<.*\>)*', l).groups()[0]
                   for l in lib.replace(" ", "").split('|')])
    lib_part = set([
        p.strip() for p in set(re.split("_|-|\.|:| ", lib))
        if len(p) > 2 and p not in delete_parts and not p.isdigit()
    ])
    lib_part.add(lib)
    return lib_part

    
p = 'Switch from slang2 to ncurses5.\n\nAddresses-Debian-Bug: 581631 Signed-off-by: LaMont Jones <lamont@debian.org>'
old_lib = 'libslang2-dev'
new_lib = 'libncurses5'

for old_part in split_lib_name_c(old_lib):
    for new_part in split_lib_name_c(new_lib):
        print(is_migration(p, old_part, new_part))

(False, '', '')
(True, 'switch from slang2 to ncurses5', 'key B A')
